# 🌳 Cuaderno 02 — Newton y Lagrange

**Métodos Numéricos · Unidad 4: Ajuste de curvas e interpolación**

Tema 4.2. Dos formas del **mismo** polinomio, más la estimación del error.

In [ ]:
#@title ⚙️ Entorno — ejecuta esta celda PRIMERO
# ---------------------------------------------------------------
# En tu computadora usa los archivos locales, sin repositorio.
# En Google Colab pide UNA vez el enlace de tu fork, lo guarda en tu
# Drive y a partir de ahí lo reutiliza solo.
# ---------------------------------------------------------------
import json
import os
import subprocess
import sys

CUADERNO = 'u4_02_newton_lagrange'
TAREA_ID = 'U4_02_Newton_Lagrange'
REPO_NOMBRE = "U4_Repo"
GUARDADO = os.path.join("CursoMN", "u4_repo.json")   # relativo a MyDrive

try:
    import google.colab  # noqa: F401
    ES_COLAB = True
except ImportError:
    ES_COLAB = False


def _raiz_local():
    """Sube desde la carpeta actual hasta encontrar lib/robomat.py."""
    candidatos = []
    nb = globals().get("__vsc_ipynb_file__")
    if nb:
        candidatos.append(os.path.dirname(os.path.abspath(nb)))
    candidatos.append(os.getcwd())
    for base in candidatos:
        b = base
        for _ in range(6):
            if os.path.exists(os.path.join(b, "lib", "robomat.py")):
                return b
            padre = os.path.dirname(b)
            if padre == b:
                break
            b = padre
    return None


def _normaliza(enlace):
    enlace = (enlace or "").strip().rstrip("/")
    if enlace.endswith(".git"):
        enlace = enlace[:-4]
    if not enlace.startswith("https://github.com/"):
        raise ValueError("Necesito un enlace https://github.com/USUARIO/"
                         + REPO_NOMBRE + " (tu fork).")
    return enlace

# Repositorio del curso (upstream) y carpetas que el alumno nunca edita:
# se refrescan en cada ejecución para que el material y el motor nunca estén
# atrasados, aunque el fork se haya quedado viejo.
CURSO = "https://github.com/ITH-MGL-MN/" + REPO_NOMBRE
MATERIAL = ("lib", "grader", "config", "docs", "ejercicios", "tools")


def _git(*args):
    """Corre git dentro del clon. Devuelve (funcionó, salida)."""
    r = subprocess.run(["git", "-C", REPO] + list(args),
                       capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr).strip()


def actualizar():
    """Trae tus commits de GitHub y refresca el material del curso.

    El clon de Drive es desechable: tus respuestas viven en el cuaderno.
    Primero se actualiza el fork (lo que editaste en GitHub) y después se
    copia el material del curso desde `upstream/main`. Si tienes cambios
    locales en el clon, no se tocan: se avisa y se sigue.
    """
    ok, _ = _git("rev-parse", "--is-inside-work-tree")
    if not ok:
        print("⚠️ La carpeta no es un repositorio git: uso los archivos tal cual.")
        return
    _git("remote", "add", "upstream", CURSO + ".git")
    ok, salida = _git("fetch", "--all", "--prune", "--quiet")
    if not ok:
        print("⚠️ No pude conectar con GitHub; sigo con lo que hay en Drive.")
        return

    _git("checkout", "--", *MATERIAL)          # limpia el refresco anterior
    _, rama = _git("rev-parse", "--abbrev-ref", "HEAD")
    rama = rama or "main"
    ok, salida = _git("merge", "--ff-only", "origin/" + rama)
    if ok:
        print("Fork        : al día con tus commits de GitHub")
    else:
        print("⚠️ Tu fork no avanza en línea recta (tienes commits propios")
        print("   o cambios locales): lo dejo como está y solo refresco el")
        print("   material del curso.")

    ok, salida = _git("checkout", "upstream/main", "--", *MATERIAL)
    if ok:
        _git("reset", "--quiet")
        _, sha = _git("rev-parse", "--short", "upstream/main")
        print("Material    : actualizado al curso", sha)
    else:
        print("⚠️ No pude refrescar el material del curso:", salida.splitlines()[:1])


if not ES_COLAB:
    REPO = _raiz_local()
    if REPO is None:
        raise RuntimeError(
            "No encontré la raíz del repositorio (lib/robomat.py).\n"
            "Abre el cuaderno dentro de U4_Repo o ejecútalo desde esa carpeta.")
    print("Modo        : local (sin repositorio)")
    print("Repositorio :", REPO)
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
    ruta_json = os.path.join(DRIVE, GUARDADO)

    guardado = {}
    if os.path.exists(ruta_json):
        try:
            with open(ruta_json, encoding="utf-8") as f:
                guardado = json.load(f)
        except (OSError, json.JSONDecodeError) as exc:
            raise RuntimeError(
                f"No se pudo leer la configuracion de Drive: {ruta_json}"
            ) from exc

    enlace = (guardado.get("repo") or "").strip()
    if enlace:
        print("✅ Repositorio guardado:", enlace)
        print("   (si cambió, edita la caja de abajo y pulsa el botón)")
    else:
        try:
            enlace = input("Pega el enlace de TU fork de U4_Repo: ")
        except Exception:
            enlace = ""
        enlace = (enlace or "").strip()
        if not enlace:
            raise RuntimeError(
                "Sin repositorio no puedo continuar en Colab.\n"
                "Haz fork del repositorio del curso y pega su enlace.")

    enlace = _normaliza(enlace)
    REPO = os.path.join(DRIVE, "CursoMN", REPO_NOMBRE)

    if not os.path.exists(REPO):
        print("Clonando tu repositorio…")
        subprocess.run(["git", "clone", enlace + ".git", REPO], check=True)
    actualizar()

    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump({"repo": enlace}, f, indent=1)
    print("Repositorio :", REPO)
    print("Actualizar  : vuelve a ejecutar esta celda cuando subas cambios a GitHub")

# Instala dependencias del repositorio solo si falta alguna.
import importlib.util

_modulos_requeridos = ("numpy", "scipy", "sympy", "matplotlib", "yaml", "requests")
_faltantes = [nombre for nombre in _modulos_requeridos
              if importlib.util.find_spec(nombre) is None]
if _faltantes:
    requisitos = os.path.join(REPO, "requirements.txt")
    if not os.path.isfile(requisitos):
        raise FileNotFoundError("Falta requirements.txt en el repositorio.")
    print("Instalando dependencias faltantes:", ", ".join(_faltantes))
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", requisitos],
                   check=True)

# Indica al motor empaquetado dónde está la configuración del repositorio.
os.environ["MN_REPO"] = REPO

# --- Motor de calificación -------------------------------------------
# Se usa el código fuente si está (rama `test`, para depurar); si no, el blob
# empaquetado de grader/, que es lo que recibe el alumno.
_ruta_modulos = os.path.join(REPO, "lib", "mn_motor")
_ruta_blob = os.path.join(REPO, "grader", CUADERNO + ".txt")

_lib = os.path.join(REPO, "lib")
if _lib not in sys.path:
    sys.path.insert(0, _lib)          # ajuste_curvas y, si está, mn_motor

if os.path.isfile(os.path.join(_ruta_modulos, "__init__.py")):
    import mn_motor as _motor
    globals().update({n: getattr(_motor, n) for n in _motor.__all__})
    print("Motor       : módulos de lib/mn_motor (modo profesor)")
else:
    import base64
    import zlib

    if not os.path.exists(_ruta_blob):
        raise FileNotFoundError(
            "No encuentro el motor: falta grader/" + CUADERNO + ".txt\n"
            "Clona el repositorio completo (incluye la carpeta grader/).")
    with open(_ruta_blob, encoding="utf-8") as f:
        codigo = zlib.decompress(base64.b64decode(f.read().strip())).decode("utf-8")
    _MOTOR = {"__name__": "mn_motor_blob", "__file__": "<blob>"}
    exec(compile(codigo, "<blob>", "exec"), _MOTOR)
    globals().update({n: _MOTOR[n] for n in _MOTOR["__all__"]})
    print("Motor       : blob verificado de grader/")

usar_cuaderno(CUADERNO, TAREA_ID)

# --- Preludio estilo MATLAB y herramientas de gráficas ---------------
with open(os.path.join(REPO, "lib", "robomat.py"), encoding="utf-8") as f:
    exec(f.read(), globals())

if REPO not in sys.path:
    sys.path.insert(0, REPO)          # para importar ejercicios/
sys.path.append(os.path.join(REPO, "tools"))
import importlib
import visualizar
importlib.reload(visualizar)

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 110
print("Preludio    : lib/robomat.py · tools/visualizar.py")
print("\nListo. Ejecuta la celda de identificación.")

In [ ]:
#@title 👤 Identificación y semilla reproducible de preguntas
import os

if ES_COLAB:
    import requests
    from google.colab import auth
    import google.auth
    import google.auth.transport.requests

    auth.authenticate_user()
    creds, _ = google.auth.default()
    creds.refresh(google.auth.transport.requests.Request())
    respuesta = requests.get(
        "https://www.googleapis.com/drive/v3/about?fields=user",
        headers={"Authorization": "Bearer " + creds.token},
        timeout=20,
    )
    respuesta.raise_for_status()
    cuenta = respuesta.json().get("user", {})
    email = cuenta.get("emailAddress", "").strip()
    nombre = cuenta.get("displayName", "")
else:
    email = os.environ.get("MN_EMAIL", "").strip()
    if not email:
        email = input("Escribe tu correo institucional @tecnm.mx: ").strip()
    nombre = ""

if not (email and "@" in email and email.lower().endswith(".tecnm.mx")):
    raise ValueError("Usa tu correo institucional terminado en .tecnm.mx.")

MI_TAREA = generar_tarea(email, nombre)
print("Alumno:", email)
print("NC / semilla reproducible:", extraer_nc(email), "/", MI_TAREA["semilla"])
print("Cuaderno:", MI_TAREA["cuaderno"], "·", len(MI_TAREA["preguntas"]), "preguntas")


---

## Método 1 · Tabla de diferencias divididas

La primera diferencia dividida es una pendiente,

$$f[x_i,x_j]=\frac{f(x_i)-f(x_j)}{x_i-x_j} \tag{18.12}$$

y las siguientes se construyen con las anteriores:

$$f[x_i,x_j,x_k]=\frac{f[x_i,x_j]-f[x_j,x_k]}{x_i-x_k} \tag{18.13}$$

La tabla es triangular. En esta disposición, los coeficientes del polinomio de Newton quedan en la **diagonal**:

| $i$ | $x_i$ | $f(x_i)$ | Primera . . . | Segunda . . . . . . | Tercera . . . . . . . . . . |
|---:|---|---|---|---|---|
| 0 | $x_0$ | $\mathbf{f(x_0)}$ | | | |
| 1 | $x_1$ | $f(x_1)$ | $\mathbf{f[x_1,x_0]}$ | | |
| 2 | $x_2$ | $f(x_2)$ | $f[x_2,x_1]$ | $\mathbf{f[x_2,x_1,x_0]}$ | |
| 3 | $x_3$ | $f(x_3)$ | $f[x_3,x_2]$ | $f[x_3,x_2,x_1]$ | $\mathbf{f[x_3,x_2,x_1,x_0]}$ |

Con los coeficientes de la diagonal $b_0 = f(x_0)$, $b_1 = f[x_1,x_0]$, $b_2 = f[x_2,x_1,x_0]$, $\dots$, $b_n = f[x_n,x_{n-1},\dots,x_0]$, se construye el **polinomio de interpolación de Newton en diferencias divididas**:

$$f_n(x) = b_0 + b_1(x - x_0) + b_2(x - x_0)(x - x_1) + \cdots + b_n(x - x_0)(x - x_1)\cdots(x - x_{n-1}) \tag{18.15}$$

Fíjate en el patrón para armarlo sin memorizar toda la fórmula:
* Cada coeficiente $b_k$ multiplica al producto de los factores $(x - x_i)$ desde $x_0$ hasta $x_{k-1}$.
* El último nodo ($x_n$) **nunca aparece** en un factor $(x - x_n)$; solo se usa para calcular el último coeficiente $b_n$.

### Ejemplo 18.3 · construir la tabla paso a paso

Usaremos los datos en el orden indicado para $f(x)=\ln(x)$:
$$\begin{align}
(x_0,f(x_0))=&(1,0),\\
(x_1,f(x_1))=&(4,1.386294),\\
(x_2,f(x_2))=&(6,1.791759) \text{ y} \\
(x_3,f(x_3))=&(5,1.609438).
\end{align}$$
**No ordenes los nodos por su tamaño**: cada índice conserva el par que le corresponde, aunque aquí $x_3=5$ sea menor que $x_2=6$.

**Paso 1: primeras diferencias.** Divide la diferencia de los valores de cada par consecutivo entre la diferencia de sus $x$:

$$f[x_1,x_0]=\frac{1.386294-0}{4-1}=0.462098$$

$$f[x_2,x_1]=\frac{1.791759-1.386294}{6-4}=0.2027325$$

$$f[x_3,x_2]=\frac{1.609438-1.791759}{5-6}=0.182321$$

Observa el último cociente: numerador y denominador son negativos, por eso la pendiente es positiva.

**Paso 2: segundas diferencias.** Resta dos primeras diferencias vecinas y divide entre la distancia entre los nodos extremos de esos tres datos:

$$f[x_2,x_1,x_0]=\frac{0.2027325-0.462098}{6-1}=-0.0518731$$

$$f[x_3,x_2,x_1]=\frac{0.182321-0.2027325}{5-4}=-0.0204115$$

El denominador es entre los extremos ($6-1$ y $5-4$), no entre los dos nodos del primer par.

**Paso 3: tercera diferencia.** Aplica la misma regla a las dos segundas diferencias:

$$f[x_3,x_2,x_1,x_0]=\frac{-0.0204115-(-0.0518731)}{5-1}=0.0078654$$

La tabla se llena por filas: cada nueva columna usa los dos valores contiguos de la columna anterior. Las celdas vacías indican que ya no hay suficientes nodos para calcular otra diferencia:

| $i$ | $x_i$ | $f(x_i)$ | 1.ª diferencia | 2.ª diferencia | 3.ª diferencia |
|---:|---:|---:|---:|---:|---:|
| 0 | 1 | **0** | | | |
| 1 | 4 | 1.386294 | **0.462098** | | |
| 2 | 6 | 1.791759 | 0.2027325 | **-0.0518731** | |
| 3 | 5 | 1.609438 | 0.182321 | -0.0204115 | **0.0078654** |

**Paso 4: lee los coeficientes.** Los $b_k$ son la diagonal que empieza en la esquina superior izquierda: $b_0=f(x_0)$, $b_1=f[x_1,x_0]$, $b_2=f[x_2,x_1,x_0]$ y $b_3=f[x_3,x_2,x_1,x_0]$. Por tanto, $b_0=0$, $b_1=0.462098$, $b_2=-0.0518731$ y $b_3=0.0078654$. Estos son los coeficientes del polinomio de Newton; el último también permite estimar el error al pasar de grado 2 a grado 3.

![Comparación de las interpolaciones lineal, cuadrática y cúbica de ln 2 (figura 18.6 del libro)](https://raw.githubusercontent.com/ITH-MGL-MN/U4_Repo/main/docs/figuras/libro/fig18-6_ln2_cubica.jpg)

*Figura 18.6. La cúbica sigue mejor la curva, aunque más grado no siempre mejora la aproximación en todo el intervalo.*

In [ ]:
#@title ✏️ Actividad a mano · Tabla de diferencias divididas
EJ_1 = mano_enunciado("NEWTON_TABLA")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Tabla de diferencias divididas
# Reporta los números que obtuviste, con 4 decimales.
b = [0.0, 0.0, 0.0, 0.0]
mano_comprueba("NEWTON_TABLA", b)

### 💻 Programa la función

Abre `ejercicios/newton_lagrange/tabla_diferencias.py` en GitHub o copia la plantilla aquí abajo.

In [ ]:
#@title 💻 Tu función · Tabla de diferencias divididas
from ejercicios.newton_lagrange.tabla_diferencias import *

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("NEWTON_TABLA")

In [ ]:
#@title 📊 Laboratorio · Tabla de diferencias divididas
visualizar.laboratorio("NEWTON_TABLA", EJ_1)

In [ ]:
#@title 🧪 Pregunta 1 · Tabla de diferencias divididas
# Cuando tu función ya funcione, contéstala aquí.
pregunta(1)

---

## Método 2 · Polinomio de Newton en diferencias divididas

La forma general agrega un término por nodo:

$$f_n(x)=b_0+b_1(x-x_0)+\cdots+b_n(x-x_0)\cdots(x-x_{n-1}) \tag{18.7}$$

Se evalúa con Horner: $p=b_n$ y luego $p\leftarrow p(x-x_k)+b_k$. Esta función
**debe llamar** a `coeficientes_newton` y a `evaluar_newton`.

In [ ]:
#@title ✏️ Actividad a mano · Polinomio de Newton en diferencias divididas
EJ_2 = mano_enunciado("NEWTON_EVAL")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Polinomio de Newton en diferencias divididas
# Reporta los números que obtuviste, con 4 decimales.
# Puedes escribir aquí el código de los pasos
x = 4.5
y = 0.0
mano_comprueba("NEWTON_EVAL", y)

### 💻 Programa la función

Abre `ejercicios/newton_lagrange/interpolacion_newton.py` en GitHub o copia la plantilla aquí abajo.
El método completo debe **llamar** a sus piezas: no repitas la cuenta.

In [ ]:
#@title 💻 Tu función · Polinomio de Newton en diferencias divididas
from ejercicios.newton_lagrange.interpolacion_newton import *

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("NEWTON_EVAL")

In [ ]:
#@title 📊 Laboratorio · Polinomio de Newton en diferencias divididas
visualizar.laboratorio("NEWTON_EVAL", EJ_2)

In [ ]:
#@title 🧪 Pregunta 2 · Polinomio de Newton en diferencias divididas
# Cuando tu función ya funcione, contéstala aquí.
pregunta(2)

---

## Método 3 · Polinomio de Lagrange

Es una reformulación que evita las diferencias divididas:

$$\begin{align}
f_n(x)=&\sum_{i=0}^{n}L_i(x)f(x_i),\tag{18.20}\\
L_i(x)=&\prod_{\substack{j=0\\ j\neq i}}^{n}\frac{x-x_j}{x_i-x_j}. \tag{18.21}
\end{align}$$

Cada peso vale 1 en su nodo y 0 en los demás. Da **el mismo** polinomio que
Newton.

![Pesos de Lagrange en cada nodo y suma que forma el polinomio interpolante (figura 18.10 del libro)](https://raw.githubusercontent.com/ITH-MGL-MN/U4_Repo/main/docs/figuras/libro/fig18-10_lagrange.jpg)

*Figura 18.10. Cada peso $L_i(x)$ vale 1 en su nodo y 0 en los demás; la suma de los términos es el polinomio interpolante.*

### Ejemplo 18.6 · pesos en $x=2$

Para los nodos $x_0=1$, $x_1=4$, $x_2=6$ y valores $f(x_0)=0$, $f(x_1)=1.386294$, $f(x_2)=1.791760$:

| $i$ | $L_i(2)$ . . . . . . . . . . . . . .  . . . . . . . . . . . . . . | $L_i(2)f(x_i)$ . . . . . . . . |
|---:|---:|---:|
| 0 | $\dfrac{(2-4)(2-6)}{(1-4)(1-6)}=0.5333$ | 0 |
| 1 | $\dfrac{(2-1)(2-6)}{(4-1)(4-6)}=0.6667$ | 0.924196 |
| 2 | $\dfrac{(2-1)(2-4)}{(6-1)(6-4)}=-0.2000$ | -0.358352 |
| **Suma** | **1.0000** | **0.565844** |

Los pesos suman 1, por lo que el polinomio reproduce cualquier función constante.

### Newton y Lagrange: ¿cuándo conviene cada forma?

| Criterio | Newton | Lagrange |
|---|---|---|
| Cálculo previo | Tabla de diferencias divididas, $O(n^2)$ | Ninguno |
| Agregar un nodo | Se agrega un término | Se recalculan los pesos |
| Estimar el error | Directamente con un punto extra | Requiere calcular diferencias divididas |
| Uso típico | Explorar grados y estimar error | Una interpolación de grado fijo |

In [ ]:
#@title ✏️ Actividad a mano · Polinomio de Lagrange
EJ_3 = mano_enunciado("LAGRANGE")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Polinomio de Lagrange
# Reporta los números que obtuviste, con 4 decimales.
x = 3
y = 0.0
mano_comprueba("LAGRANGE", [0.0])

### 💻 Programa la función

Abre `ejercicios/newton_lagrange/interpolacion_lagrange.py` en GitHub o copia la plantilla aquí abajo.
El método completo debe **llamar** a sus piezas: no repitas la cuenta.

In [ ]:
#@title 💻 Tu función · Polinomio de Lagrange
from ejercicios.newton_lagrange.interpolacion_lagrange import *

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("LAGRANGE")

In [ ]:
#@title 📊 Laboratorio · Polinomio de Lagrange
visualizar.laboratorio("LAGRANGE", EJ_3)

In [ ]:
#@title 🧪 Pregunta 3 · Polinomio de Lagrange
# Cuando tu función ya funcione, contéstala aquí.
pregunta(3)

---

## Método 4 · Error del polinomio

El término que se agrega al subir de grado es una estimación del error
de grado $n$:

$$R_n=f_{n+1}(x)-f_n(x)=b_n\prod_{i<n}(x-x_i) \tag{18.19}$$

### Ejemplo 18.4 · estimación y error verdadero

Para la aproximación de $\ln 2$ con la parábola y el punto extra del ejemplo 18.3:

| Cantidad | Valor |
|---|---:|
| Estimación del error, $R_2$ | 0.0629242 |
| Error verdadero | 0.1273028 |

La estimación con un nodo extra puede **subestimar** el error verdadero; úsala como referencia, no como garantía ni como único criterio de paro.

In [ ]:
#@title ✏️ Actividad a mano · Error del polinomio
EJ_4 = mano_enunciado("ERROR_NEWTON")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Error del polinomio
# Reporta los números que obtuviste, con 4 decimales.
mano_comprueba("ERROR_NEWTON", [0.0])

### 💻 Programa la función

Abre `ejercicios/newton_lagrange/error_newton.py` en GitHub o copia la plantilla aquí abajo.
El método completo debe **llamar** a sus piezas: no repitas la cuenta.

In [ ]:
#@title 💻 Tu función · Error del polinomio
from ejercicios.newton_lagrange.error_newton import *

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("ERROR_NEWTON")

In [ ]:
#@title 📊 Laboratorio · Error del polinomio
visualizar.laboratorio("ERROR_NEWTON", EJ_4)

In [ ]:
#@title 🧪 Pregunta 4 · Error del polinomio
# Cuando tu función ya funcione, contéstala aquí.
pregunta(4)

In [ ]:
#@title ✅ 1 · Verificación de los ejercicios a mano
mano_comprueba_todos(["NEWTON_TABLA", "NEWTON_EVAL", "LAGRANGE", "ERROR_NEWTON"])

In [ ]:
#@title 📝 2 · Verificación de todas las preguntas (no envía nada)
puntos, maximo = calificar()

In [ ]:
#@title 📤 3 · Envío de las respuestas
enviar(email)

In [ ]:
#@title 🔎 4 · Verificación de lo que ya está enviado (no envía nada)
consultar_calificacion()